# 📦 SmishGuard — Complete Dataset Merge (FINAL)

Run this notebook FIRST before training. It combines all datasets into one clean `combined_smishing.csv`.

---

## 📥 Upload These Files to Colab Before Running

| File to Upload | Download From | Required? |
|---|---|---|
| `spam.csv` | Already have it | ✅ YES |
| `Dataset_5971.csv` | https://data.mendeley.com/datasets/f45bkkt8pr/1 | ✅ YES |
| `Dataset_10191.csv` | https://data.mendeley.com/datasets/vmg875v4xs/1 | ✅ YES |
| `imc2025_smishing.csv` | https://github.com/reportsmishing/Smishing-Dataset-IMC25 | ⭐ Recommended |

**How to upload**: In Colab, click the 📁 folder icon on the left → drag and drop all files.

---

## ✅ Verified Column Structures (Already Handled)
| File | Columns | Labels |
|---|---|---|
| spam.csv | v1, v2 | ham / spam |
| Dataset_5971.csv | LABEL, TEXT | Ham / Spam / Smishing |
| Dataset_10191.csv | LABEL, TEXT, URL, EMAIL, PHONE | Ham / Spam / Smishing |
| imc2025_smishing.csv | varies | all smishing |


In [ ]:
# ================================================================
# CELL 1 — IMPORTS
# ================================================================
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

print('✅ Ready')
print()
print('Files found in /content:')
for f in sorted(os.listdir('/content')):
    if f.endswith('.csv'):
        size = os.path.getsize(f'/content/{f}') // 1024
        print(f'  📄 {f}  ({size} KB)')

In [ ]:
# ================================================================
# CELL 2 — INSPECT ALL UPLOADED FILES
# Run this to verify columns before merging
# ================================================================
files_to_check = [
    '/content/spam.csv',
    '/content/Dataset_5971.csv',
    '/content/Dataset_10191.csv',
    '/content/imc2025_smishing.csv',
]

print('FILE INSPECTION REPORT')
print('=' * 60)
for fpath in files_to_check:
    if os.path.exists(fpath):
        try:
            df_peek = pd.read_csv(fpath, encoding='latin-1', nrows=2)
            print(f'✅ {os.path.basename(fpath)}')
            print(f'   Columns : {list(df_peek.columns)}')
            print(f'   Row 1   : {df_peek.iloc[0].to_dict()}')
        except Exception as e:
            print(f'⚠️  {os.path.basename(fpath)} — error: {e}')
    else:
        print(f'❌ {os.path.basename(fpath)} — NOT UPLOADED (will be skipped)')
    print()

In [ ]:
# ================================================================
# CELL 3 — LOAD DATASET 1: UCI SMS Spam
# Columns: v1 (label), v2 (text)
# Labels : ham / spam
# ================================================================
all_frames = []

try:
    df1 = pd.read_csv('/content/spam.csv', encoding='latin-1')[['v1', 'v2']]
    df1.columns = ['label', 'text']
    df1['label'] = df1['label'].str.lower().str.strip()
    df1['source'] = 'uci'
    df1 = df1[df1['label'].isin(['spam', 'ham'])]
    all_frames.append(df1)

    vc = df1['label'].value_counts()
    print(f'✅ Dataset 1 — UCI SMS Spam')
    print(f'   Rows  : {len(df1)}')
    print(f'   HAM   : {vc.get("ham", 0)}')
    print(f'   SPAM  : {vc.get("spam", 0)}')

except FileNotFoundError:
    print('❌ spam.csv NOT FOUND — this is required. Please upload it.')
except Exception as e:
    print(f'❌ UCI load error: {e}')

In [ ]:
# ================================================================
# CELL 4 — LOAD DATASET 2: Mishra & Soni
# Download : https://data.mendeley.com/datasets/f45bkkt8pr/1
# Columns  : LABEL, TEXT (+ other attribute columns)
# Labels   : Ham / Spam / Smishing  →  we map Smishing → spam
# ================================================================
try:
    df2 = pd.read_csv('/content/Dataset_5971.csv', encoding='latin-1')

    # Normalize column names to lowercase
    df2.columns = [c.strip().lower() for c in df2.columns]

    # This dataset is confirmed to have 'label' and 'text' columns
    df2 = df2[['label', 'text']].copy()
    df2['label'] = df2['label'].str.lower().str.strip()

    # Map: smishing → spam (both are malicious — binary classifier)
    df2['label'] = df2['label'].replace({'smishing': 'spam', 'legitimate': 'ham'})
    df2 = df2[df2['label'].isin(['spam', 'ham'])]
    df2['source'] = 'mishra_soni'
    all_frames.append(df2)

    vc = df2['label'].value_counts()
    print(f'✅ Dataset 2 — Mishra & Soni Smishing Dataset')
    print(f'   Rows  : {len(df2)}')
    print(f'   HAM   : {vc.get("ham", 0)}')
    print(f'   SPAM  : {vc.get("spam", 0)}  (includes original spam + smishing)')

except FileNotFoundError:
    print('⏭️  Dataset_5971.csv not uploaded — skipping')
    print('   Download: https://data.mendeley.com/datasets/f45bkkt8pr/1')
except KeyError as e:
    print(f'⚠️  Column not found: {e}')
    print('   Run Cell 2 to inspect actual column names')
except Exception as e:
    print(f'⚠️  Mishra load error: {e}')

In [ ]:
# ================================================================
# CELL 5 — LOAD DATASET 3: LLM Balanced Smishing Dataset
# Download : https://data.mendeley.com/datasets/vmg875v4xs/1
# Columns  : LABEL, TEXT, URL, EMAIL, PHONE
# Labels   : Ham / Spam / Smishing  (3,397 each — perfectly balanced)
# ================================================================
try:
    df3 = pd.read_csv('/content/Dataset_10191.csv', encoding='latin-1')
    df3.columns = [c.strip().lower() for c in df3.columns]

    # Confirmed columns: label, text, url, email, phone
    # We only need label and text for training
    df3 = df3[['label', 'text']].copy()
    df3['label'] = df3['label'].str.lower().str.strip()

    # Map smishing → spam
    df3['label'] = df3['label'].replace({'smishing': 'spam', 'legitimate': 'ham'})
    df3 = df3[df3['label'].isin(['spam', 'ham'])]
    df3['source'] = 'llm_balanced'
    all_frames.append(df3)

    vc = df3['label'].value_counts()
    print(f'✅ Dataset 3 — LLM Balanced Smishing Dataset')
    print(f'   Rows  : {len(df3)}')
    print(f'   HAM   : {vc.get("ham", 0)}')
    print(f'   SPAM  : {vc.get("spam", 0)}  (spam + smishing merged)')

except FileNotFoundError:
    print('⏭️  Dataset_10191.csv not uploaded — skipping')
    print('   Download: https://data.mendeley.com/datasets/vmg875v4xs/1')
except KeyError as e:
    print(f'⚠️  Column not found: {e}')
    print('   Run Cell 2 to inspect actual column names')
except Exception as e:
    print(f'⚠️  LLM Balanced load error: {e}')

In [ ]:
# ================================================================
# CELL 6 — LOAD DATASET 4: IMC 2025
# Download : https://github.com/reportsmishing/Smishing-Dataset-IMC25
# File     : dataset/final_dataset_output.csv  (inside the ZIP)
# Upload as: imc2025_smishing.csv
# Note     : This dataset is ALL smishing — no ham rows
# ================================================================
try:
    df4 = pd.read_csv('/content/imc2025_smishing.csv', encoding='latin-1')
    df4.columns = [c.strip().lower() for c in df4.columns]

    print(f'   Columns found: {list(df4.columns)}')

    # Find the text column (column name varies in this dataset)
    text_col = next(
        (c for c in df4.columns
         if any(k in c for k in ['message', 'text', 'body', 'sms', 'content'])),
        None
    )
    label_col = next(
        (c for c in df4.columns
         if any(k in c for k in ['label', 'class', 'type', 'category'])),
        None
    )

    if text_col:
        if label_col:
            # Has label column
            df4 = df4[[label_col, text_col]].copy()
            df4.columns = ['label', 'text']
            df4['label'] = df4['label'].str.lower().str.strip()
            label_map = {
                'smishing'    : 'spam',
                'phishing'    : 'spam',
                'spam'        : 'spam',
                '1'           : 'spam',
                'ham'         : 'ham',
                'legitimate'  : 'ham',
                'non-smishing': 'ham',
                '0'           : 'ham',
            }
            df4['label'] = df4['label'].map(label_map)
        else:
            # No label column — entire dataset is smishing
            df4 = df4[[text_col]].copy()
            df4.columns = ['text']
            df4['label'] = 'spam'
            print('   No label column found — treating all rows as spam (smishing)')

        df4 = df4.dropna(subset=['label', 'text'])
        df4 = df4[df4['label'].isin(['spam', 'ham'])]
        df4['source'] = 'imc2025'
        all_frames.append(df4)

        vc = df4['label'].value_counts()
        print(f'✅ Dataset 4 — IMC 2025 Smishing Dataset')
        print(f'   Rows  : {len(df4)}')
        print(f'   HAM   : {vc.get("ham", 0)}')
        print(f'   SPAM  : {vc.get("spam", 0)}')
    else:
        print(f'⚠️  Could not find text column in IMC2025')
        print(f'   Available columns: {list(df4.columns)}')
        print('   Manually set text_col above and re-run this cell')

except FileNotFoundError:
    print('⏭️  imc2025_smishing.csv not uploaded — skipping')
    print('   Download ZIP from: https://github.com/reportsmishing/Smishing-Dataset-IMC25')
    print('   Then upload the file: dataset/final_dataset_output.csv as imc2025_smishing.csv')
except Exception as e:
    print(f'⚠️  IMC2025 load error: {e}')

In [ ]:
# ================================================================
# CELL 7 — MERGE + CLEAN + DEDUPLICATE
# ================================================================
if not all_frames:
    raise RuntimeError('❌ No datasets loaded. Upload at least spam.csv')

print(f'Datasets loaded: {len(all_frames)}')
for f in all_frames:
    print(f'  {f["source"].iloc[0]:20} → {len(f):>6} rows')
print()

# ── Step 1: Merge ────────────────────────────────────────────────
df = pd.concat(all_frames, ignore_index=True)
print(f'After merge              : {len(df):>6} rows')

# ── Step 2: Basic clean ──────────────────────────────────────────
df['text']  = df['text'].astype(str).str.strip()
df['label'] = df['label'].astype(str).str.lower().str.strip()
df = df[df['text'].str.len() > 5]          # Remove very short rows
df = df[df['label'].isin(['spam', 'ham'])]  # Keep only valid labels
df.dropna(subset=['text', 'label'], inplace=True)
print(f'After basic clean        : {len(df):>6} rows')

# ── Step 3: Deduplicate ──────────────────────────────────────────
# Same message in multiple datasets should count only once
before = len(df)
df['_text_norm'] = df['text'].str.lower().str.strip().str.replace(r'\s+', ' ', regex=True)
df = df.drop_duplicates(subset=['_text_norm'], keep='first')
df = df.drop(columns=['_text_norm'])
print(f'Duplicates removed       : {before - len(df):>6}')
print(f'After deduplication      : {len(df):>6} rows')

# ── Step 4: Final shuffle ────────────────────────────────────────
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# ── Step 5: Report ───────────────────────────────────────────────
vc = df['label'].value_counts()
print()
print('=' * 50)
print('FINAL COMBINED DATASET')
print('=' * 50)
print(f'Total rows : {len(df):>6}')
print(f'HAM        : {vc.get("ham",  0):>6}  ({vc.get("ham",  0)/len(df)*100:.1f}%)')
print(f'SPAM       : {vc.get("spam", 0):>6}  ({vc.get("spam", 0)/len(df)*100:.1f}%)')
print()
print('Contribution by source:')
src = df.groupby('source')['label'].value_counts().unstack(fill_value=0)
print(src.to_string())
print('=' * 50)

In [ ]:
# ================================================================
# CELL 8 — QUALITY VISUALIZATION
# ================================================================
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('SmishGuard — Combined Dataset Quality Report', fontsize=13, fontweight='bold')

# 1. Class balance
colors = ['#2196F3', '#F44336']
vc.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 30, f'{v:,}', ha='center', fontweight='bold')

# 2. Source breakdown
src_counts = df['source'].value_counts()
src_colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#00BCD4']
src_counts.plot(kind='bar', ax=axes[1],
                color=src_colors[:len(src_counts)], edgecolor='black', width=0.6)
axes[1].set_title('Rows Per Dataset Source')
axes[1].set_xlabel('')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=25)
for i, v in enumerate(src_counts.values):
    axes[1].text(i, v + 10, f'{v:,}', ha='center', fontweight='bold', fontsize=9)

# 3. Message length distribution
df['_len'] = df['text'].str.len()
axes[2].hist(df[df['label'] == 'ham']['_len'],
             bins=50, alpha=0.6, color='#2196F3', label='HAM')
axes[2].hist(df[df['label'] == 'spam']['_len'],
             bins=50, alpha=0.6, color='#F44336', label='SPAM')
axes[2].set_title('Message Length Distribution')
axes[2].set_xlabel('Characters')
axes[2].set_ylabel('Count')
axes[2].legend()
df.drop(columns=['_len'], inplace=True)

plt.tight_layout()
plt.savefig('dataset_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: dataset_quality.png')

In [ ]:
# ================================================================
# CELL 9 — SAVE COMBINED DATASET
# ================================================================

# Save with source column (keeps traceability)
df.to_csv('/content/combined_smishing.csv', index=False)
print(f'✅ Saved: combined_smishing.csv')
print(f'   Total rows : {len(df):,}')
print(f'   File size  : {os.path.getsize("/content/combined_smishing.csv")//1024:,} KB')
print()
print('NEXT STEP — Open SmishGuard_v2_FINAL.ipynb')
print('In Cell 6, replace the UCI load block with:')
print()
print('  df = pd.read_csv("/content/combined_smishing.csv")')
print('  df = df[["label", "text"]]   # drop source column')
print('  print(f"Loaded: {len(df)} rows")')

# Download
from google.colab import files
files.download('/content/combined_smishing.csv')
files.download('/content/dataset_quality.png')
print()
print('✅ Download started')